# Properties


```{contents}
:local:
:depth: 2
```


```{index} Properties
```



```{index} Properties
```

## Properties




A **property** is a class member that a caller reads and writes like a field, while the class decides what each read or write actually does. Properties appear as public data members, but they are implemented as small methods called *accessors*: `get` runs when the property is read, `set` runs when it is assigned. That is why C# code almost never exposes a public field: a property gives the same convenient syntax and keeps the class in control.

The forms you will meet, as patterns:

```csharp
public string Phone { get; set; }            // auto-implemented: the compiler stores the value
public string Name { get; }                  // read-only: assigned only in the constructor
public decimal Total { get; private set; }   // readable everywhere, assignable only inside the class
public string Summary => $"{Name} ({Phone})";   // computed: recalculated on every read
public string Phone                          // full property with a backing field
{
    get { return phone; }
    set { phone = value; }                   // value is the assigned value
}
```

Inside a `set` accessor the keyword `value` names whatever the caller assigned. The runnable example uses the first, second, and fourth forms:


In [ ]:
using System;

Contact first = new Contact("Marie Ortiz", "773-508-7890");
Console.WriteLine($"{first.Name}: {first.Phone}");
first.Phone = "555-555-5555";          // allowed: Phone has a public set
Console.WriteLine($"{first.Name}: {first.Phone}");
Console.WriteLine(first.Summary);      // computed each time it is read

class Contact
{
    public string Name { get; }                      // read-only after construction
    public string Phone { get; set; }                // readable and writable
    public string Summary => $"{Name} ({Phone})";   // computed property

    public Contact(string name, string phone)
    {
        Name = name;
        Phone = phone;
    }
}


The output is `Marie Ortiz: 773-508-7890`, then `Marie Ortiz: 555-555-5555`, then `Marie Ortiz (555-555-5555)`. Reading `first.Phone` looks exactly like reading a field, and assigning it looks like assigning a field, yet both go through accessors. Adding the line `first.Name = "Someone Else";` would not compile, because `Name` has no `set`; the read-only form is how a class promises that a value fixed at construction stays fixed. `Summary` has no storage of its own: the `=>` form recomputes the string from the current `Name` and `Phone` every time, which is why it shows the new phone number without being updated.

The next three subsections show what properties replace: the explicit getter and setter methods that older code, and languages such as Java, use for the same purpose. Reading them explains what the accessors above do behind the scenes, and the subsection after them rewrites the same class with properties.


The four forms differ in who may read and write, which is the design decision behind each one:

| Form | Read from outside | Write from outside | Typical use |
| --- | --- | --- | --- |
| `{ get; set; }` | yes | yes | a plain data field the caller may change, such as a phone number |
| `{ get; }` | yes | only in the constructor | an identity fixed at creation, such as a name or SKU |
| `{ get; private set; }` | yes | no; the class changes it through methods | a running total that `Add` and `Clear` maintain |
| `=> expression` | yes, recomputed each time | no | a value derived from other properties, such as a summary line |
| full `get`/`set` with a backing field | yes | yes, through validation | a value that must be checked before it is stored |


```{index} OOP; getter
```


```{index} Getters
```



```{index} Getters
```

(getters)=

### Getters




In instance methods you have an extra way of getting data in and out of the method:
Reading or setting instance variables (fields). The simplest methods do nothing but reading
or setting instance variables. We start with those:

The `private` in front
of the field declarations was important to keep code outside the
class from messing with the values. On the other hand we do want
others to be able to *inspect* the name, phone and email,
so how do we do that? Use **public methods**.

Since the fields are accessible anywhere *inside* the class’s instance methods,
and public methods can be used from *outside* the class, we can simply code

```csharp
      public string GetName()
      {
         return name;
      }

      public string GetPhone()
      {
         return phone;
      }

      public string GetEmail()
      {
         return email;
      }
```

These methods allow one-way communication of the name, phone and email
values out from the object.
These are examples
of a simple category of methods: A *getter* simply returns the value of a part
of the object’s state, without changing the object at all.

Note again that there is no `static` in the method heading.
The field value for the *current* Contact is returned.

A standard convention that we are following:
Have getter methods names start with “Get”,
followed by the name of the data to be returned.

In this first simple version of Contact we add one further method, to
print all the contact information with labels.

```csharp
      public void Print()
      {
         Console.WriteLine (@"Name:  {0}
Phone: {1}
Email: {2}", name, phone, email);
      }
```

Again, we use the instance variable names, plugging them into a format string.
Remember the `@` syntax
for multiline strings gives us verbatim string literals that are interpreted literally
without escape sequences.

You can see the entire Contact class, and a program that tests it, here:

````{dropdown} Complete code: Contact (version 1)
{file}`contact1.cs`:


```csharp
using System;
namespace IntroCS
{
/// A simple class / object example:
   class Contact // nothing in here is marked static
   {
      private string name, phone, email;

      // a constructor - no return type
      public Contact(string fullName, string phoneNumber, string emailAddress)
      {
         name = fullName;
         phone = phoneNumber;
         email = emailAddress;
      }

      // getter methods
      public string GetName()
      {
         return name;
      }

      public string GetPhone()
      {
         return phone;
      }

      public string GetEmail()
      {
         return email;
      }

      ///print with labels
      public void Print()
      {
         Console.WriteLine (@"Name:  {0}
Phone: {1}
Email: {2}", name, phone, email);
      }
   }               // end Print chunk
}
```

{file}`test_contact1.cs`:


```csharp
using System;

namespace IntroCS
{
   public class TestContact
   {
      public static void Main()
      {
         Contact c1 = new Contact("Marie Ortiz", "773-508-7890",
                                  "mortiz2@luc.edu");
         Contact c2 = new Contact("Otto Heinz", "773-508-9999",
                                  "oheinz@luc.edu");
         Console.WriteLine("Marie's full name: " + c1.GetName());
         Console.WriteLine("Her phone number: " + c1.GetPhone());
         Console.WriteLine("Her email: " + c1.GetEmail());
         Console.WriteLine("\nFull contact info for Otto:");
         c2.Print();
      }
   }
}
```
````

This is your first complete class defining a new type of object.
Look carefully to get used to the features introduced, before we add
more ideas:

```{index} class; Contact version 2
```

Run the same class as top-level statements. The three getters let the caller inspect the fields; nothing outside the class can assign to them.


In [ ]:
using System;

Contact c1 = new Contact("Marie Ortiz", "773-508-7890", "mortiz2@example.edu");
Contact c2 = new Contact("Otto Heinz", "773-508-9999", "oheinz@example.edu");
Console.WriteLine("Marie's full name: " + c1.GetName());
Console.WriteLine("Her phone number: " + c1.GetPhone());
Console.WriteLine("Her email: " + c1.GetEmail());
Console.WriteLine("\nFull contact info for Otto:");
c2.Print();

class Contact
{
    private string name, phone, email;

    public Contact(string fullName, string phoneNumber, string emailAddress)
    {
        name = fullName;
        phone = phoneNumber;
        email = emailAddress;
    }

    public string GetName() { return name; }
    public string GetPhone() { return phone; }
    public string GetEmail() { return email; }

    public void Print()
    {
        Console.WriteLine($"Name:  {name}\nPhone: {phone}\nEmail: {email}");
    }
}


Each getter returns the current value of one private field. Try adding `Console.WriteLine(c1.name);` before the class: it does not compile, because `name` is private, and that refusal is the point of the design. The caller gets exactly the access the class chose to offer.


```{index} OOP; this keyword
```



```{index} This Object
```

### This Object




We will be making an elaboration on the Contact class from here on. We introduce
new parts individually, but the whole code is here:

````{dropdown} Complete code: Contact (version 2)
{file}`contact2.cs`:


```csharp
using System;
namespace IntroCS
{
/// Elaborated class / object example with setters and ToString, use this
   class Contact
   {
      private string name, phone, email;

      // a constructor - no return type
      public Contact(string name, string phone, string email)
      {
         this.name = name;
         this.phone = phone;
         this.email = email;
      }

      // getter methods
      public string GetName()
      {
         return name;
      }

      public string GetPhone()
      {
         return phone;
      }

      public string GetEmail()
      {
         return email;
      }

      // setter methods
      public void SetPhone(string newPhone)
      {
         phone = newPhone;
      }
      // SetEmail chunk
      public void SetEmail(string email)
      {
         this.email = email;
      }
      // ToString chunk
      public override string ToString()
      {
         return string.Format (@"Name:  {0}
Phone: {1}
Email: {2}", name, phone, email);
      }
      // new Print chunk
      ///print with labels
      public void Print()  // use ToString
      {
         Console.WriteLine(ToString());
      }
   }
}
```

{file}`test_contact2.cs`:


```csharp
using System;
using System.Collections.Generic;

namespace IntroCS
{
   public class TestContact2
   {  // main chunk
      public static void Main()
      {
         Contact c1 = new Contact("Marie Ortiz", "773-508-7890",
                                  "mortiz2@luc.edu");
         Contact c2 = new Contact("Otto Heinz", "773-508-9999",
                                  "oheinz@luc.edu");
         Console.WriteLine("Marie's full name: " + c1.GetName());
         Console.WriteLine("Her phone number: " + c1.GetPhone());
         Console.WriteLine("Her email: " + c1.GetEmail());
         Console.WriteLine("All together:\n{0}", c1);
         Console.WriteLine("Full contact info for Otto:");
         c2.Print();
         c1.SetEmail("maria.ortiz@gmail.com");
         c2.SetPhone("123-456-7890");
         Contact c3 = new Contact("Amy Li", "847-111-2222",
                                  "amy.li22@yahoo.com");
         Console.WriteLine("With changes and added contact:");
         var allc = new List<Contact>(new Contact[] {c1, c2, c3});
         foreach(Contact c in allc) {
            Console.WriteLine("\n"+c);
         }
      }
   }                                       // end of chunk
}
```
````

The current object is *implicit* inside a constructor or instance method definition,
but it can be referred to *explicitly*. It is called `this`.
In a constructor or instance method, `this` is automatically a legal
local variable to reference.
You usually do not need to
use it explicitly, but you could. For example the current `Contact` object’s
`name` field could be referred to as either `this.name` or the shorter plain `name`.
In our next version of the Contact class we will see several places where
an explicit `this` is useful.

In the first version of the constructor, repeated here,

```csharp
      public Contact(string fullName, string phoneNumber, string emailAddress)
      {
         name = fullName;
         phone = phoneNumber;
         email = emailAddress;
      }
```

we used different names for the instance variables and the formal
parameter names that we used to initialize the instance variables. We
chose reasonable names, but we are adding extra names that we are not going
to use later, and it can be confusing. The most obvious names for the formal
parameters that will initialize the instance variables are the *same* names.

If we are not careful, there is a problem with that. An instance variable,
however named, and a local variable are not the same. This is
nonsensical:

```csharp
public Contact(string name, string phone, string email)
{
   name = name; // ????
   ...
```

Logically we want this pseudo-code in the constructor:

> instance variable `name` `=` local variable `name`

We have to disambiguate the two uses. The compiler always looks for
*local* variable identifiers *first*, so plain `name` will refer to the local
variable `name` declared in the formal parameter list.
This local variable identifier *hides* the matching instance variable
identifier. We have to do something else to refer to the instance variable.
The explicit `this` object comes to the rescue: `this.name` refers to
a part of this object. It must refer to the
instance variable, not the local variable. Our alternate constructor is:

```csharp
      public Contact(string name, string phone, string email)
      {
         this.name = name;
         this.phone = phone;
         this.email = email;
      }
```

The runnable form shows `this` doing its job: the parameter and the field share a name, and `this.name` picks the field.


In [ ]:
using System;

Contact c = new Contact("Amy Li", "847-111-2222");
c.Print();

class Contact
{
    private string name, phone;

    public Contact(string name, string phone)
    {
        this.name = name;      // the field on the left, the parameter on the right
        this.phone = phone;
    }

    public void Print()
    {
        Console.WriteLine($"{name}: {phone}");
    }
}


The output is `Amy Li: 847-111-2222`. Remove `this.` from the two assignments and run again: the output becomes `: ` because each statement assigns the parameter to itself and the fields keep their default `null`. Inside `Print`, plain `name` is unambiguous, since no local variable hides it there.


```{index} Setters
```



```{index} Setters
```

### Setters




The original version of Contact makes a Contact object be
*immutable*: Once it is created with the constructor, there is no way
to change its internal state. The only assignments to the private
instance variables are the ones in the constructor. In real life
people can change their email address. We might like to allow that
with our Contact objects. Users can read the data in a Contact with the
*getter* methods. Now we need *setter* methods. The naming conventions
are similar: start with “Set”. In this case we must supply the new data,
so setter methods need a parameter:

```csharp
      public void SetPhone(string newPhone)
      {
         phone = newPhone;
      }
```

In `SetPhone`, like in our original constructor, we illustrate using a
*new* name for the parameter that sets the instance variable. For
comparison we use the alternate identifier matching approach in the
other setter:

```csharp
      public void SetEmail(string email)
      {
         this.email = email;
      }
```

Now we can alter the contents of a Contact:

```csharp
Contact c1 = new Contact("Marie Ortiz", "773-508-7890", "mortiz2@luc.edu");
c1.SetEmail ("maria.ortiz@gmail.com");
c1.SetPhone("555-555-5555");
c1.Print();
```

would print

``` none
Name:  Marie Ortiz
Phone: 555-555-5555
Email: maria.ortiz@gmail.com
```

The complete second version, run as top-level statements, combines the setters with the `ToString` override developed in the next section:


In [ ]:
using System;

Contact c1 = new Contact("Marie Ortiz", "773-508-7890", "mortiz2@example.edu");
c1.SetEmail("maria.ortiz@example.com");
c1.SetPhone("555-555-5555");
c1.Print();
Console.WriteLine();
Console.WriteLine(c1);          // Console.WriteLine calls ToString for you

class Contact
{
    private string name, phone, email;

    public Contact(string name, string phone, string email)
    {
        this.name = name;
        this.phone = phone;
        this.email = email;
    }

    public string GetName() { return name; }
    public void SetPhone(string newPhone) { phone = newPhone; }
    public void SetEmail(string email) { this.email = email; }

    public override string ToString()
    {
        return $"Name:  {name}\nPhone: {phone}\nEmail: {email}";
    }

    public void Print()
    {
        Console.WriteLine(ToString());
    }
}


Both printed blocks show the changed phone and email, because the setters assigned the private fields and every later read sees the new values. A setter can also refuse a change, which a public field never can; the next subsection shows that with a property.


```{index} Properties Replace Getters and Setters
```

(properties-replace-accessors)=

### Properties Replace Getters and Setters


`GetPhone` and `SetPhone` work, but C# gives the pair one name and the field syntax. A full property keeps the private backing field and puts the two accessors together; the `set` accessor can validate `value` before storing it, and a property with only `get` is read-only. The caller writes `c.Phone` to read and `c.Phone = "..."` to write, and never sees the field.

```csharp
private string phone = "";          // backing field

public string Phone                  // the property the caller uses
{
    get { return phone; }
    set
    {
        if (string.IsNullOrWhiteSpace(value))
            throw new ArgumentException("Phone cannot be blank.");
        phone = value;
    }
}
```


In [ ]:
using System;

Contact c = new Contact("Marie Ortiz", "773-508-7890");
Console.WriteLine(c.Phone);
c.Phone = "555-555-5555";          // runs the set accessor
Console.WriteLine(c.Phone);
try
{
    c.Phone = "";                  // the set accessor rejects this
}
catch (ArgumentException ex)
{
    Console.WriteLine($"Rejected: {ex.Message}");
}
Console.WriteLine(c.Phone);        // unchanged by the rejected assignment

class Contact
{
    private string phone = "";

    public string Name { get; }

    public string Phone
    {
        get { return phone; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Phone cannot be blank.");
            phone = value;
        }
    }

    public Contact(string name, string phone)
    {
        Name = name;
        Phone = phone;     // runs the setter's check during construction too
    }
}


The output is `773-508-7890`, `555-555-5555`, `Rejected: Phone cannot be blank.`, and `555-555-5555` again. The blank assignment never reached the field, so the object stayed valid. The constructor assigns through the property on purpose, so the same check protects construction; assigning `phone = ""` directly inside the class would bypass it. When no check is needed, the auto-implemented form `public string Phone { get; set; }` is the whole declaration; add the full form only when a `set` has work to do.


```{index} Guided Practice: Validate an Email Property
```

### Guided Practice: Validate an Email Property


Add an `Email` property to `Contact` with a backing field. The setter must reject a value that does not contain `@` and store any accepted value in lowercase. Expected output: `mortiz2@example.edu`, then `Rejected: Email must contain @.`, then `mortiz2@example.edu` again.


In [ ]:
using System;

Contact c = new Contact("Marie Ortiz", "MOrtiz2@Example.edu");
Console.WriteLine(c.Email);
try { c.Email = "not-an-address"; }
catch (ArgumentException ex) { Console.WriteLine($"Rejected: {ex.Message}"); }
Console.WriteLine(c.Email);

class Contact
{
    private string email = "";

    public string Name { get; }

    public string Email
    {
        get { return email; }
        set
        {
            // Your code starts here.
            email = value;
            // Your code ends here.
        }
    }

    public Contact(string name, string email)
    {
        Name = name;
        Email = email;
    }
}


In [ ]:
using System;

// Solution
Contact c = new Contact("Marie Ortiz", "MOrtiz2@Example.edu");
Console.WriteLine(c.Email);
try { c.Email = "not-an-address"; }
catch (ArgumentException ex) { Console.WriteLine($"Rejected: {ex.Message}"); }
Console.WriteLine(c.Email);

class Contact
{
    private string email = "";

    public string Name { get; }

    public string Email
    {
        get { return email; }
        set
        {
            if (!value.Contains('@'))
                throw new ArgumentException("Email must contain @.");
            email = value.ToLowerInvariant();
        }
    }

    public Contact(string name, string email)
    {
        Name = name;
        Email = email;
    }
}


The starter prints the address exactly as typed and then accepts the bad value, so its second and third lines differ from the expected output. In the solution, lowercase normalization happens in one place, the setter, so every path that assigns an email gets the same rule.


```{index} ToString; override
```


```{index} ToString Override
```



```{index} ToString Override
```

(tostring)=

## ToString Override




All object in C# have a `ToString` method because all types inherit from the base class
System.Object. It converts an object to its string representation for display. It is used
to perform internal string concatenation and Write.

```{index} Default ToString behavior
```



```{index} Default ToString behavior
```

### Default ToString behavior




By default, ToString returns returns the `fully qualified name` of the type of the Object. For example:

In [1]:
using System;
namespace IntroCSCS
{
   class Animal
   {
      public string name;

      public Animal(string name)
      {
         this.name = name;
      }
   }

   public class TestAnimal
   {
      public static void Main()
      {
         Animal frog = new Animal("Froggy");
         Console.WriteLine(frog.ToString()); ///// print: IntroCSCS.Animal; Animal is the type
         Console.WriteLine(frog.name);       ///// print: Froggy
      }
   }
}

IntroCSCS.Animal
Froggy


```{index} Overriding ToString
```



```{index} Overriding ToString
```

### Overriding ToString




Think more generally about string representations: all the built-in type values can be concatenated into strings with the `+` operator, or displayed with `Console.Write`. We would like that behavior with our custom types, too.

Types frequently `override` the `Object.ToString` method to provide a more suitable string representation of a particular type instead of the default fully qualified type name. For example:


In [1]:
using System;
    class Employee
    {
       public string Name { get; set; }
       public int Salary { get; set; }

       public override string ToString()
       {
          return "Employee: " + Name + " " + Salary;      ///// returns object data
       }
    }

    class Test
    {
       public static void Main()
       {
          Employee employee = new Employee { Name = "John", Salary = 100000 };
          Console.WriteLine(employee.ToString());   ///// print: Employee: John 100000
          Console.WriteLine(employee);              ///// print: Employee: John 100000
                                                    /////        as ToString is the default output

       }
    }


Employee: John 100000
Employee: John 100000


See what the ToString method does now: it uses the object state (e.g., the values of variables name and
salary in this case) and implicitly covert and *return* a single string representation of the object.

```{index} Print() Customization
```



```{index} Print() Customization
```

### Print() Customization




You can further customize the ToString overriding. For example, in the preceding `Employee` class,
in addition to the following `ToString` override, you might like to have a convenience method
`Print` using `ToString`. We want one instance method, `Print` to call another instance
method `ToString` for the *same* object:

```csharp
public void Print()
{
   Console.WriteLine(ToString());
   // Console.WriteLine(this);         ///// "this" returns the same as ToString()
}
```

Now take a look at the test and output:

```csharp
Employee employee = new Employee { Name = "John", Salary = 100000 };
Console.WriteLine(employee.ToString());
Console.WriteLine(employee);
employee.Print();
```

```text
Employee: John 100000
Employee: John 100000
Employee: John 100000
```

All three lines are the same because `Console.WriteLine(employee)` and `Print` both end up calling the one `ToString` override.



% When we use `Console.WriteLine` on this current object, which is *not*

% already a string, there is an automatic call to `ToString`.

% .. index:: redeclaring instance variables error

% compiler error; before error in text

% instance variable; redeclaring error

% .. \_local-variables-hiding-instance-variables:

% Local Variables Hiding Instance Variables

% ——————————————

% A common error is for students to try to declare the instance variables twice,

% once in the regular instance variable declarations,

% *outside of any constructor or method* and then again *inside* a

% constructor, like::

% public Contact(string fullName, string phoneNumber, string emailAddress)

% {

% string name = fullName; // LOGICAL ERROR!

% string phone = phoneNumber; // LOGICAL ERROR!

% string email = emailAddress; // LOGICAL ERROR

% }

% This is deadly. It is worse than redeclaring a local variable, which at least will

% trigger a compiler error.

% .. warning::

% Instance variable *only* get declared outside of all

% functions and constructors. Same-name

% local variable declarations hide the

% instance variables, but *compile just fine*. The local variables disappear

% after the constructor ends, leaving the instance variables

% *without* your desired initialization. Instead the hidden instance variables

% just get the default initialization, `null` for an object

% or 0 for a number.

% There is a related strange compiler error. This is not likely to happen frequently,

% but thinking through its logic (or illogic) could be helpful in understanding

% local and instance variables:

% Generally when you get a compiler error, the error is at or *before* the location the

% error is referenced, but with local variables covering instance variables,

% the real cause can come later in the text of the method. Below, when you first

% refer to `r` in `Badnames`, it appears to be correctly referring to the

% instance variable `r`::

% class ForwardError

% {

% private int r = 3;

% // …

% void BadNames(int a, int b)

% {

% int n = a*r + b; // legal in text* just\* to here; instance field r

% //…

% int r = a % b; // r declaration makes *earlier* line wrong

% //…

% }

% The compiler scans through *all* of `BadNames`, and sees the `r` declared locally

% in its scope.

% The error may be marked on the earlier line, where the compiler then assumes

% `r` is the later declared local `int` variable, not the instance variable.

% The error it sees is a local variable used before declaration.

% This is based on a real student example.

% This example points to a second issue: using variable names that

% that are too short and not descriptive of the variable meaning, and so may

% easily be the same name as something unrelated.

% .. index:: lifetime; vs. scope

% scope; vs. lifetime

% Lifetime and Scope Exercise/Example

% ~

% Be careful to distinguish lifetime and scope. Either a local variable or

% an instance variable can be temporarily out of scope, but still be alive.

% Can you construct an example to illustrate that? One of ours is

% :repsrc:`lifetime_scope/lifetime_scope.cs`.


```{rubric} Footnotes
```
